In [ ]:
# Install necessary packages if running in Google Colab.
!pip install pymupdf
!pip install pypdf
!pip install orjsonl

In [ ]:
# Working with .pdf and .txt documents.
from pypdf import PdfReader
import re

# Handling .JSONL objects.
import orjsonl

In [ ]:
class ArraysToJSONL:
    """
    Convert list of ID-tagged problems and solutions to .JSONL file.

    Attributes:
        prompts_completions: dict of problems and solutions, indexed by ID.
        json_array: list of .JSON objects.
    """
    # Attribute types
    prompts_completions: dict[str, list[str]]
    json_array: list[dict]

    def __init__(self,
                 problems_array: list[str],
                 solutions_array: list[str],
                 spacer: str = "-",
                 alt_id_pattern: str = None) -> None:
        """
        Initialize the converter and perform conversions.

        Precondition: spacer follows regex formatting.

        Default ID pattern is Chapter Number + "-" + Question Number.
        spacer allows for a different delimiter between chapter and question
        numbers.
        alt_id_pattern allows for specification of an irregular ID pattern.
        """
        if alt_id_pattern is not None:
            id_pattern = alt_id_pattern
        else:
            id_pattern = "[1-9][0-9]*" + spacer + "[1-9][0-9]*"
        self.prompts_completions = self.__match_problems_and_solutions(
            problems_array, solutions_array, id_pattern)
        self.json_array = self.__get_json()

    def save_jsonl(self, filename: str) -> None:
        """Save the JSON array to a .JSONL file."""
        orjsonl.save(filename, self.json_array)

    def show_json(self) -> None:
        """Print all JSONs in the JSON array."""
        for json in self.json_array:
            print(json)

    def __match_problems_and_solutions(self,
                                       problems_array: list[str],
                                       solutions_array: list[str],
                                       id_pattern: str) -> dict[str, list[str]]:
        """Match problems and solutions based on ID."""
        out = {}
        for problem in problems_array:
            question_id = re.search(id_pattern, problem)
            if question_id is not None:
                prompt = problem[question_id.end() + 1:]
                question_id = question_id.group(0)
                out[question_id] = [prompt]
        for solution in solutions_array:
            question_id = re.search(id_pattern, solution)
            if question_id is not None:
                completion = solution[question_id.end() + 1:]
                question_id = question_id.group(0)
                # Only append the solution if a matching problem exists.
                if id in self.prompts_completions:
                    out[question_id].append(completion)
        return out


    def __get_json(self) -> list[dict]:
        """Convert the prompts_completions dict to a list of JSON objects."""
        out = []
        for question_id in self.prompts_completions:
            # Check that there is both a non-blank prompt and completion.
            if (len(self.prompts_completions[question_id]) == 2 and
                self.prompts_completions[question_id][0] != '' and
                self.prompts_completions[question_id][1] != ''):
                json = {
                    "prompt" : self.prompts_completions[question_id][0],
                    "completion" : self.prompts_completions[question_id][1]
                }
                out.append(json)
        return out

In [ ]:
# Execute this block of code if you have no modifications to make.
# Convert the .pdf file to a .txt file.
text = ""
reader = PdfReader('textbook.pdf')  # Change the filename to filename of doc.

for page in reader.pages:
    page_text = page.extract_text()
    text += page_text

with open("textbook.txt", "w") as my_file: # Change filename as required.
    my_file.write(text)

In [ ]:
# Execute this block of code if you wish to remove the header of each page.
# Convert the .pdf file to a .txt file.
text = ""
reader = PdfReader('textbook.pdf') # Change the filename to filename of doc.

for page in reader.pages:
    all_page_text = page.extract_text()
    page_text_index = re.search(".*", all_page_text)
    if page_text_index is None:
        continue
    page_text = all_page_text[page_text_index.end():]
    text += page_text

with open("textbook.txt", "w") as my_file: # Change filename as required.
    my_file.write(text)

In [ ]:
# Find the problems and solutions sections and save them to .txt document.
# BEFORE RUNNING: Open the .txt file from one of two prior blocks and place
# markers to specify start and end of problem and solutions sections.
# MARKERS: @----PROBLEMS START----@
#          @----PROBLEMS END----@
#          @----SOLUTIONS START----@
#          @----SOLUTIONS END----@
my_file = open("textbookmarked.txt", "r") # Use marked .txt filename here.
text = my_file.read()

def find_text_between_markers(all_text: str, marker: str) -> str:
    """Find text between two markers in the textbook."""
    out = ""
    start_marker = "@----" + marker + " START----@"
    end_marker = "@----" + marker + " END----@"

    split_by_start_array = re.split(start_marker, all_text)
    for below_start_marker_section in split_by_start_array:
        split_by_end_array = re.split(end_marker, below_start_marker_section)
        if len(split_by_end_array) != 1:
            out += split_by_end_array[0]
    return out

problems_text = find_text_between_markers(text, marker = "PROBLEMS")
solutions_text = find_text_between_markers(text, marker = "SOLUTIONS")

with open("problems.txt", "w") as my_file: # Change filename as required.
    my_file.write(problems_text)

with open("solutions.txt", "w") as my_file: # Change filename as required.
    my_file.write(solutions_text)

In [ ]:
# Split the problems and solutions text into an array of individual problems and
# solutions, keeping the ID at the head.
problems_file = open("problems.txt", "r")
solutions_file = open("solutions.txt", "r")
problems_text = problems_file.read()
solutions_text = solutions_file.read()

problems_array_tb = re.split("PROBLEM", problems_text)
solutions_array_tb = re.split("SOLUTION", solutions_text)

# "\\." is "." in regex formatting.
jsonl = ArraysToJSONL(problems_array_tb, solutions_array_tb, spacer = "\\.")
jsonl.show_json()
jsonl.save_jsonl("textbook_name_prompt_completion.jsonl")